# Lab 16: DQN on CartPole

**Module 16** | Read `notes/16-dqn.pdf` before running this notebook.

Train a DQN agent with Stable-Baselines3 on CartPole-v1 and plot episode rewards.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Deep Q-Networks on CartPole

CartPole-v1 is a classic control benchmark: keep a pole balanced on a cart by pushing left or right.
The state is four continuous values (cart position, velocity, pole angle, angular velocity); the agent chooses one of two discrete actions.

**Deep Q-Networks (DQN)** approximate the action-value function $Q(s,a)$ with a neural network.
Experience replay and a target network stabilize training when bootstrapping from the network's own predictions.
**Stable-Baselines3 (SB3)** is a Python library that wraps these RL details in ready-made algorithms so you can focus on environment setup, monitoring, and evaluation.

In this lab you will:

1. Wrap the environment with `Monitor` so episode rewards are logged automatically.
2. Train a DQN policy for **30,000 environment steps** and print episode statistics during training.
3. Plot the reward curve and run **5 deterministic test episodes** to inspect actions and totals.
4. Compare your results to the CartPole **solved threshold** (average reward of 195 over 100 episodes).


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Reinforcement learning (RL)** | An agent learns by taking actions in an environment and receiving numeric rewards. The goal is to maximize total reward over time. |
| **Q-value / action-value Q(s,a)** | The expected total future reward if you take action a in state s and then follow your policy. |
| **Deep Q-Network (DQN)** | Uses a neural network to approximate Q(s,a) for many states. The policy picks the action with the highest predicted Q-value. |
| **Experience replay buffer** | Stores past (state, action, reward, next state) transitions and samples random mini-batches for training. Breaks correlation between consecutive steps and reuses data efficiently. |
| **Target network** | A copy of the Q-network updated less often. Stabilizes DQN by preventing the target from chasing a moving prediction. |
| **Epsilon-greedy exploration** | Mostly pick the best known action, but sometimes pick a random action to discover better strategies. |
| **Stable-Baselines3 (SB3)** | A library of tested RL algorithms (DQN, A2C, PPO, etc.). `model.learn()` runs the full training loop: collect data, update the network, log progress. |
| **Gymnasium environment** | A standard API: `reset()` starts an episode, `step(action)` returns the next state, reward, and done flag. |
| **Monitor wrapper** | Records episode rewards and lengths so you can plot learning curves after training. |
| **MlpPolicy** | SB3 policy type: feeds the raw state vector through a multi-layer perceptron (MLP) neural network. |

Refer back to this table whenever a term appears in code or output.


### Environment setup

**What this cell does:** Imports libraries, defines a training callback that prints episode stats, and creates the CartPole environment wrapped in `Monitor`.

**Why we do this:** You need a logged environment before training so rewards are saved automatically. The callback gives live feedback every few episodes so you can see learning progress without waiting for training to finish.


**What to look for in the output**

After running, you should see:
- `CartPole-v1 environment ready`
- Observation space described as a 4-dimensional Box (four state numbers)
- Action space described as Discrete(2) (0=left, 1=right)
- The solved threshold printed as 195


In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

# Project root (parent of the labs/ folder)
ROOT = Path("..").resolve()

# Human-readable names for the two discrete actions
ACTION_NAMES = {0: "left", 1: "right"}

# Official CartPole-v1 "solved" criterion: average reward >= 195 over 100 consecutive episodes
SOLVED_THRESHOLD = 195


class EpisodeStatsCallback(BaseCallback):
    """Print episode reward and length as training progresses."""

    def __init__(self, print_every: int = 5):
        super().__init__()
        self.print_every = print_every  # print every N completed episodes
        self.episode_count = 0

    def _on_step(self) -> bool:
        # SB3 calls this after every environment step during model.learn()
        for info in self.locals.get("infos", []):
            # Monitor adds an "episode" key when an episode finishes
            if info and "episode" in info:
                self.episode_count += 1
                ep = info["episode"]
                if self.episode_count % self.print_every == 0:
                    print(
                        f"  Training episode {self.episode_count}: "
                        f"reward={ep['r']:.0f}, length={ep['l']}"
                    )
        return True  # return True to keep training


# Create the CartPole environment and wrap it with Monitor for episode logging
env = Monitor(gym.make("CartPole-v1"))
obs_space = env.observation_space
act_space = env.action_space

print("CartPole-v1 environment ready")
print(f"  Observation space: {obs_space} (4 state variables)")
print(f"  Action space: {act_space} (0=left, 1=right)")
print(f"  Solved threshold: average reward >= {SOLVED_THRESHOLD} over 100 episodes")


### Train the DQN agent

**What this cell does:** Creates a DQN model with `MlpPolicy` and calls `model.learn()` for 30,000 timesteps.

**Why we do this:** This is where SB3 does the heavy lifting. Internally it:
- Collects transitions by interacting with CartPole (exploration via epsilon-greedy)
- Stores transitions in a **replay buffer**
- Samples mini-batches and updates the Q-network with gradient descent
- Periodically syncs a **target network** for stable targets
- Uses `verbose=1` to print SB3's own training progress table


**What to look for in the output**

During training you should see:
- `Starting DQN training for 30,000 timesteps...`
- SB3 progress lines (fps, timesteps, mean episode reward when available)
- Callback lines every 5 episodes like `Training episode 5: reward=..., length=...`
- Rewards should generally climb from low values (often 20 to 40) toward higher values
- `Training finished.` at the end


In [ ]:
TOTAL_TIMESTEPS = 30_000

# DQN("MlpPolicy", env): SB3 builds an MLP that outputs Q-values for each action
# verbose=1 tells SB3 to print its internal training statistics
model = DQN("MlpPolicy", env, verbose=1)
stats_callback = EpisodeStatsCallback(print_every=5)

print(f"Starting DQN training for {TOTAL_TIMESTEPS:,} timesteps...")
# learn() runs the full DQN loop: explore, store, replay, update weights
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=stats_callback)
print("Training finished.")


### Evaluate training and test the policy

**What this cell does:** Summarizes all training episodes, plots the reward curve, runs 5 greedy test episodes, and compares results to the solved threshold.

**Why we do this:** Training rewards show whether learning happened. Test episodes use `deterministic=True` (no random exploration) to show what the final policy actually does. Comparing to 195 tells you if the agent meets the classic CartPole benchmark.


**What to look for in the output**

You should see:
- A TRAINING SUMMARY block with episode counts, first/last/max rewards, and means
- A matplotlib plot with episode rewards, a moving average line, and a dashed line at 195
- Five DETERMINISTIC TEST EPISODES with steps, action previews, and total rewards near 500 if training succeeded
- A COMPARISON block with test mean reward and solved status (if 100+ training episodes were logged)


In [ ]:
# Monitor stored these lists during training
rewards = env.get_episode_rewards()
episode_lengths = env.get_episode_lengths()

print("=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Episodes completed during training: {len(rewards)}")
print(f"First episode reward: {rewards[0]:.0f}")
print(f"Last episode reward: {rewards[-1]:.0f}")
print(f"Max episode reward: {max(rewards):.0f}")
print(f"Overall mean reward: {np.mean(rewards):.2f}")
if len(rewards) >= 10:
    mean_last_10 = float(np.mean(rewards[-10:]))
    print(f"Mean reward (last 10 episodes): {mean_last_10:.2f}")
if len(rewards) >= 100:
    mean_last_100 = float(np.mean(rewards[-100:]))
    print(f"Mean reward (last 100 episodes): {mean_last_100:.2f}")
    solved = mean_last_100 >= SOLVED_THRESHOLD
    print(f"Solved (last 100 >= {SOLVED_THRESHOLD})? {solved}")
else:
    mean_last_10 = float(np.mean(rewards[-10:])) if len(rewards) >= 10 else float(np.mean(rewards))
    print(f"(Need 100 episodes for official solved check; using last 10 mean as proxy)")

# Plot raw rewards and a 10-episode moving average
plt.figure(figsize=(9, 4))
plt.plot(rewards, alpha=0.6, label="Episode reward")
if len(rewards) >= 10:
    window = np.convolve(rewards, np.ones(10) / 10, mode="valid")
    plt.plot(range(9, 9 + len(window)), window, linewidth=2, label="10-episode moving avg")
plt.axhline(SOLVED_THRESHOLD, color="gray", linestyle="--", label=f"solved threshold ({SOLVED_THRESHOLD})")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN on CartPole-v1: training rewards")
plt.legend()
plt.tight_layout()
plt.show()

print()
print("=" * 60)
print("5 DETERMINISTIC TEST EPISODES (greedy policy)")
print("=" * 60)

# Fresh test env (no Monitor needed); deterministic=True disables exploration noise
test_env = gym.make("CartPole-v1")
test_totals = []

for ep in range(5):
    obs, _ = test_env.reset()
    done = False
    step = 0
    total_reward = 0.0
    actions_taken: list[int] = []
    step_rewards: list[float] = []

    while not done:
        # deterministic=True: always pick the action with highest Q-value
        action, _ = model.predict(obs, deterministic=True)
        action_int = int(action)
        actions_taken.append(action_int)
        obs, reward, terminated, truncated, _ = test_env.step(action_int)
        total_reward += reward
        step_rewards.append(float(reward))
        step += 1
        done = terminated or truncated

    test_totals.append(total_reward)
    action_str = ", ".join(ACTION_NAMES[a] for a in actions_taken[:12])
    if len(actions_taken) > 12:
        action_str += ", ..."
    print(f"Episode {ep + 1}:")
    print(f"  Steps: {step}")
    print(f"  Actions (first 12): {action_str}")
    print(f"  Per-step rewards (first 8): {step_rewards[:8]}")
    print(f"  Total reward: {total_reward:.0f}")

test_env.close()
env.close()

print()
print("=" * 60)
print("COMPARISON TO SOLVED THRESHOLD")
print("=" * 60)
print(f"Test episode mean reward: {np.mean(test_totals):.1f}")
print(f"CartPole solved criterion: average >= {SOLVED_THRESHOLD} over 100 consecutive episodes")
if len(rewards) >= 100:
    status = "SOLVED" if mean_last_100 >= SOLVED_THRESHOLD else "NOT YET SOLVED"
    print(f"Training status (last 100 episodes): {status}")
else:
    print("Train longer (more timesteps) if rewards have not plateaued near 500.")


## Interpretation

CartPole gives +1 reward per timestep until the pole falls (max 500 steps per episode).
A healthy learning curve climbs from near-random performance (often 20 to 40) toward the maximum.

If rewards plateau early, try more timesteps or tune hyperparameters (`learning_rate`, `buffer_size`, `exploration_fraction`).

**Key takeaway:** value-based RL with function approximation can solve simple control tasks without hand-crafted features; the MLP learns useful representations directly from raw state vectors.

**What SB3 did for you:** data collection, replay buffer management, Q-network updates, target network sync, and exploration scheduling, all inside `model.learn()`.
